In [ ]:
# Ensure a clean environment
!pip uninstall -y torch torchvision torchaudio transformers trl unsloth xformers
!pip cache purge  # Clears any old package cache

# Install compatible PyTorch, Torchvision, and Xformers (CUDA 12.1)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -U xformers --index-url https://download.pytorch.org/whl/cu121

# Install required dependencies without unnecessary ones
!pip install --no-deps packaging ninja einops flash-attn trl peft accelerate bitsandbytes

# Install Unsloth from GitHub
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"

# Install additional dependencies for dataset handling
!pip install pandas openpyxl datasets huggingface_hub

# Restart the runtime to apply changes
import os
os._exit(00)


Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Found existing installation: transformers 4.50.0
Uninstalling transformers-4.50.0:
  Successfully uninstalled transformers-4.50.0
Files removed: 0
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.5/780.5 MB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 2.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 58.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.1 M

In [ ]:
import torch
import torchvision
import transformers
import trl
import unsloth
print(torch.__version__, torchvision.__version__)


<ipython-input-1-ee00a225e76d>:5: UserWarning: WARNING: Unsloth should be imported before trl, transformers to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  import unsloth


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
2.5.1+cu121 0.20.1+cu121


In [ ]:

import os
import json
import pandas as pd
from datasets import Dataset, DatasetDict
from datasets import load_dataset
from huggingface_hub import notebook_login
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

In [ ]:
# This will prompt for your HF token (make sure to use a token with write access)
notebook_login()

In [ ]:
# Run this cell to upload your Excel file
from google.colab import files
uploaded = files.upload()  # This will let you select your Hindi Q&A Excel file
excel_filename = list(uploaded.keys())[0]  # Get the filename of the uploaded file

Saving your_dataset.xlsx to your_dataset (1).xlsx


In [ ]:
# Update these with your Hugging Face username and desired dataset name
huggingface_user = "deepanshumiglani0408"  # Replace with your HF username
dataset_name = "hindi-qa-dataset"

# Load the Excel file
df_original = pd.read_excel(excel_filename)

# Display the first few rows to verify the data
print("Original Dataset Preview:")
print(df_original.head())

class Llama3HindiQADataset:
    def __init__(self, dataframe):
        self.df = dataframe
        self.prompts = []
        self.create_prompts()

    def create_prompt(self, question, answer):
        # System instruction for Hindi QA
        system_instruction = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"

        # Format in Llama 3 instruction format
        prompt = f"""<|begin_of_text|><|start_header_id|>system<|end_header_id|>{system_instruction}<|eot_id|><|start_header_id|>user<|end_header_id|>{question}<|eot_id|><|start_header_id|>assistant<|end_header_id|>{answer}<|eot_id|>"""
        return prompt

    def create_prompts(self):
        for _, row in self.df.iterrows():
            prompt = self.create_prompt(row['question'], row['answer'])
            self.prompts.append(prompt)

    def get_dataset(self):
        df = pd.DataFrame({'prompt': self.prompts})
        return df

def create_dataset_hf(dataset):
    dataset.reset_index(drop=True, inplace=True)
    return DatasetDict({"train": Dataset.from_pandas(dataset)})

# Create Llama3 formatted dataset
dataset = Llama3HindiQADataset(df_original)
df = dataset.get_dataset()

# Display a sample prompt to verify the formatting
print("\nSample formatted prompt:")
print(df['prompt'][0])

# Save and upload to Hugging Face
processed_data_path = 'processed_data'
os.makedirs(processed_data_path, exist_ok=True)

llama3_dataset = create_dataset_hf(df)
llama3_dataset.save_to_disk(os.path.join(processed_data_path, "llama3_hindi_dataset"))
llama3_dataset.push_to_hub(f"{huggingface_user}/{dataset_name}")

Original Dataset Preview:
                                            question  \
0                    आर्टिफिशियल इंटेलिजेंस क्या है?   
1  कृत्रिम बुद्धिमत्ता की दो मुख्य श्रेणियां क्या...   
2                              मशीन लर्निंग क्या है?   
3                               गहरी शिक्षा क्या है?   
4                 प्राकृतिक भाषा प्रसंस्करण क्या है?   

                                              answer  
0  आर्टिफिशियल इंटेलिजेंस उन कंप्यूटर सिस्टम के व...  
1  कृत्रिम बुद्धिमत्ता की दो मुख्य श्रेणियां संकी...  
2  मशीन लर्निंग आर्टिफिशियल इंटेलिजेंस का एक सबसे...  
3  डीप लर्निंग मशीन लर्निंग का एक सबसेट है जो कई ...  
4  प्राकृतिक भाषा प्रसंस्करण कृत्रिम बुद्धिमत्ता ...  

Sample formatted prompt:
<|begin_of_text|><|start_header_id|>system<|end_header_id|>आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।<|eot_id|><|start_header_id|>user<|end_header_id|>आर्टिफिशियल इंटेलिजेंस क्या है?<|eot_id|><|start_header_id|>assistant<|en

Saving the dataset (0/1 shards):   0%|          | 0/5000 [00:00<?, ? examples/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ?it/s]

Creating parquet from Arrow format:   0%|          | 0/5 [00:00<?, ?ba/s]

No files have been modified since last commit. Skipping to prevent empty commit.


CommitInfo(commit_url='https://huggingface.co/datasets/deepanshumiglani0408/hindi-qa-dataset/commit/e52ad3099b57967172b1b152e748378bb63e7d11', commit_message='Upload dataset', commit_description='', oid='e52ad3099b57967172b1b152e748378bb63e7d11', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/deepanshumiglani0408/hindi-qa-dataset', endpoint='https://huggingface.co', repo_type='dataset', repo_id='deepanshumiglani0408/hindi-qa-dataset'), pr_revision=None, pr_num=None)

In [ ]:
config = {
    "hugging_face_username": huggingface_user,
    "model_config": {
        "base_model": "unsloth/llama-3-8b-Instruct-bnb-4bit",  # Ensure correct model
        "finetuned_model": f"llama-3-8b-Instruct-hindi-qa-{huggingface_user}",  # Consistent naming
        "max_seq_length": 2048,
        "dtype": torch.float16,
        "load_in_4bit": True,
    },
    "lora_config": {
        "r": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "lora_alpha": 16,
        "lora_dropout": 0,
        "bias": "none",
        "use_gradient_checkpointing": True,
        "use_rslora": False,
        "use_dora": False,
        "loftq_config": None
    },
    "training_dataset": {
        "name": f"{huggingface_user}/{dataset_name}",
        "split": "train",
        "input_field": "instruction",  # Match your dataset format
    },
    "training_config": {
        "per_device_train_batch_size": 2, #2 was earlier
        "gradient_accumulation_steps": 4, #4 was earlier
        "warmup_steps": 5,
        "max_steps": -1,  # Ensures training is controlled by epochs
        "num_train_epochs": 3

        ,  # Ensure this is valid
        "learning_rate": 2e-4,
        "fp16": not torch.cuda.is_bf16_supported(),
        "bf16": torch.cuda.is_bf16_supported(),
        "logging_steps": 1,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 42,
        "output_dir": "outputs",
    }
}


In [ ]:
#average speed
config = {
    "hugging_face_username": huggingface_user,
    "model_config": {
        "base_model": "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",
        "finetuned_model": f"mistral-7b-instruct-hindi-qa-{huggingface_user}",
        "max_seq_length": 1024,  # Faster than 2048
        "dtype": torch.float16,
        "load_in_4bit": True,
    },
    "lora_config": {
        "r": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "lora_alpha": 16,
        "lora_dropout": 0,
        "bias": "none",
        "use_gradient_checkpointing": False,  # Disable for speed
        "use_rslora": False,
        "use_dora": False,
        "loftq_config": None
    },
    "training_dataset": {
        "name": f"{huggingface_user}/{dataset_name}",
        "split": "train",
        "input_field": "instruction",
    },
    "training_config": {
        "per_device_train_batch_size": 2,  # Balanced for speed & memory
        "gradient_accumulation_steps": 8,  # Faster overall training
        "warmup_steps": 5,
        "max_steps": 5000,  # Trains longer but within 4 hours
        "num_train_epochs": 1,  # Keeps training efficient
        "learning_rate": 2e-4,  # Faster learning
        "fp16": True,
        "bf16": False,
        "logging_steps": 50,  # Reduces logging overhead
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "cosine",  # Smooth learning rate decay
        "seed": 42,
        "output_dir": "outputs",
    }
}


In [ ]:
#for faster training
config = {
    "hugging_face_username": huggingface_user,
    "model_config": {
        "base_model": "unsloth/mistral-7b-instruct-v0.2-bnb-4bit",  # Ensure correct model
        "finetuned_model": f"mistral-7b-instruct-hindi-qa-{huggingface_user}",  # Consistent naming
        "max_seq_length": 1024,  # Reduced from 2048 to speed up training
        "dtype": torch.float16,
        "load_in_4bit": True,
    },
    "lora_config": {
        "r": 16,
        "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj",
                           "gate_proj", "up_proj", "down_proj"],
        "lora_alpha": 16,
        "lora_dropout": 0,
        "bias": "none",
        "use_gradient_checkpointing": True,
        "use_rslora": False,
        "use_dora": False,
        "loftq_config": None
    },
    "training_dataset": {
        "name": f"{huggingface_user}/{dataset_name}",
        "split": "train",
        "input_field": "instruction",  # Match your dataset format
    },
    "training_config": {
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 2,
        "warmup_steps": 5,
        "max_steps": 1000,
        "num_train_epochs": 3,
        "learning_rate": 1e-4,
        "fp16": True,
        "bf16": False,
        "logging_steps": 10,
        "optim": "adamw_8bit",
        "weight_decay": 0.01,
        "lr_scheduler_type": "linear",
        "seed": 42,
        "output_dir": "outputs",
    }
}


In [ ]:
# Loading the model and the tokenizer for the model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = config.get("model_config").get("base_model"),
    max_seq_length = config.get("model_config").get("max_seq_length"),
    dtype = config.get("model_config").get("dtype"),
    load_in_4bit = config.get("model_config").get("load_in_4bit"),
)

# Setup for QLoRA/LoRA peft of the base model
model = FastLanguageModel.get_peft_model(
    model,
    r = config.get("lora_config").get("r"),
    target_modules = config.get("lora_config").get("target_modules"),
    lora_alpha = config.get("lora_config").get("lora_alpha"),
    lora_dropout = config.get("lora_config").get("lora_dropout"),
    bias = config.get("lora_config").get("bias"),
    use_gradient_checkpointing = config.get("lora_config").get("use_gradient_checkpointing"),
    random_state = 42,
    use_rslora = config.get("lora_config").get("use_rslora"),
    use_dora = config.get("lora_config").get("use_dora"),
    loftq_config = config.get("lora_config").get("loftq_config"),
)

==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth 2025.3.19 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


In [ ]:
# Loading the training dataset
dataset_train = load_dataset(config.get("training_dataset").get("name"), split = config.get("training_dataset").get("split"))



In [ ]:
def formatting_func(example):
    encoding = tokenizer(
        example["prompt"],
        truncation=True,
        padding="max_length",
        max_length=config.get("model_config").get("max_seq_length")
    )
    return {
        "input_ids": encoding["input_ids"],
        "labels": encoding["input_ids"]
    }

In [ ]:
#main trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    dataset_text_field="prompt",  # Optional but fine as a fallback
    max_seq_length=config.get("model_config").get("max_seq_length"),
    dataset_num_proc=2,
    packing=False,
    formatting_func=lambda example: {
        "input_ids": tokenizer(
            example["prompt"],
            truncation=True,
            padding="max_length",
            max_length=config.get("model_config").get("max_seq_length")
        )["input_ids"],
        "labels": tokenizer(
            example["prompt"],
            truncation=True,
            padding="max_length",
            max_length=config.get("model_config").get("max_seq_length")
        )["input_ids"]
    },
    args=TrainingArguments(
        per_device_train_batch_size=config.get("training_config").get("per_device_train_batch_size"),
        gradient_accumulation_steps=config.get("training_config").get("gradient_accumulation_steps"),
        warmup_steps=config.get("training_config").get("warmup_steps"),
        max_steps=config.get("training_config").get("max_steps"),
        num_train_epochs=config.get("training_config").get("num_train_epochs"),
        learning_rate=config.get("training_config").get("learning_rate"),
        fp16=config.get("training_config").get("fp16"),
        bf16=config.get("training_config").get("bf16"),
        logging_steps=config.get("training_config").get("logging_steps"),
        optim=config.get("training_config").get("optim"),
        weight_decay=config.get("training_config").get("weight_decay"),
        lr_scheduler_type=config.get("training_config").get("lr_scheduler_type"),
        seed=42,
        output_dir=config.get("training_config").get("output_dir"),
    ),
)

Unsloth: Tokenizing ["prompt"] (num_proc=2):   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
#medium speed
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    dataset_text_field="prompt",
    max_seq_length=min(1024, config.get("model_config").get("max_seq_length")),  # Reducing max length helps with speed
    dataset_num_proc=2,  # Uses multiple CPU cores for faster preprocessing
    packing=False,  # Prevents instability in Colab
    formatting_func=lambda example: {
        "input_ids": tokenizer(
            example["prompt"],
            truncation=True,
            padding="max_length",
            max_length=min(1024, config.get("model_config").get("max_seq_length"))
        )["input_ids"],
        "labels": [
            token_id if token_id != tokenizer.pad_token_id else -100  # Mask padding tokens to prevent unnecessary updates
            for token_id in tokenizer(
                example["prompt"],
                truncation=True,
                padding="max_length",
                max_length=min(1024, config.get("model_config").get("max_seq_length"))
            )["input_ids"]
        ]
    },
    args=TrainingArguments(
        per_device_train_batch_size=max(1, config.get("training_config").get("per_device_train_batch_size") // 2),  # Reduce batch size to avoid OOM
        gradient_accumulation_steps=max(2, config.get("training_config").get("gradient_accumulation_steps")),  # Adjust accumulation steps for stability
        warmup_steps=config.get("training_config").get("warmup_steps"),
        max_steps=min(5000, config.get("training_config").get("max_steps")),  # Keep training time within 4 hours
        num_train_epochs=1,  # One epoch is enough for Colab's limit
        learning_rate=config.get("training_config").get("learning_rate"),
        fp16=True,  # Faster training
        bf16=False,  # Not supported in Colab
        logging_steps=max(50, config.get("training_config").get("logging_steps") * 2),  # Reduce logging frequency
        optim="adamw_8bit",  # Lower memory usage
        weight_decay=config.get("training_config").get("weight_decay"),
        lr_scheduler_type="cosine_with_restarts",  # Prevents sudden loss spikes
        seed=42,
        output_dir=config.get("training_config").get("output_dir"),
    ),
)


In [ ]:
#faster
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset_train,
    dataset_text_field="prompt",  # Optional but fine as a fallback
    max_seq_length=min(1024, config.get("model_config").get("max_seq_length")),  # Reduce if needed
    dataset_num_proc=1,  # Reduce CPU overhead in Colab
    packing=True,  # Enables better token usage
    formatting_func=lambda example: {
        "input_ids": tokenizer(
            example["prompt"],
            truncation=True,
            padding="max_length",
            max_length=min(1024, config.get("model_config").get("max_seq_length"))  # Adjust if needed
        )["input_ids"],
        "labels": [
            token_id if token_id != tokenizer.pad_token_id else -100  # Mask padding tokens
            for token_id in tokenizer(
                example["prompt"],
                truncation=True,
                padding="max_length",
                max_length=min(1024, config.get("model_config").get("max_seq_length"))
            )["input_ids"]
        ]
    },
    args=TrainingArguments(
        per_device_train_batch_size=max(1, config.get("training_config").get("per_device_train_batch_size") // 2),  # Lower batch size if needed
        gradient_accumulation_steps=max(2, config.get("training_config").get("gradient_accumulation_steps") // 2),  # More frequent updates
        warmup_steps=config.get("training_config").get("warmup_steps"),
        max_steps=min(300, config.get("training_config").get("max_steps")),  # Limit steps for faster runs
        num_train_epochs=1,  # Reduce to 1 epoch for quick testing
        learning_rate=max(1e-4, config.get("training_config").get("learning_rate") / 2),  # Lower for stability
        fp16=True,  # Force fp16 for speed
        bf16=False,  # Colab doesn’t support bf16
        logging_steps=max(10, config.get("training_config").get("logging_steps") * 2),  # Reduce logging overhead
        optim="adamw_torch",  # Faster optimizer
        weight_decay=config.get("training_config").get("weight_decay"),
        lr_scheduler_type="cosine",  # Faster convergence
        seed=42,
        output_dir=config.get("training_config").get("output_dir"),
    ),
)


Unsloth: Tokenizing ["prompt"]:   0%|          | 0/5000 [00:00<?, ? examples/s]

Unsloth: Hugging Face's packing is currently buggy - we're disabling it for now!


In [ ]:
print(config)  # Check if config is None


{'hugging_face_username': 'deepanshumiglani0408', 'model_config': {'base_model': 'unsloth/mistral-7b-instruct-v0.2-bnb-4bit', 'finetuned_model': 'mistral-7b-instruct-hindi-qa-deepanshumiglani0408', 'max_seq_length': 2048, 'dtype': torch.float16, 'load_in_4bit': True}, 'lora_config': {'r': 16, 'target_modules': ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'], 'lora_alpha': 16, 'lora_dropout': 0, 'bias': 'none', 'use_gradient_checkpointing': True, 'use_rslora': False, 'use_dora': False, 'loftq_config': None}, 'training_dataset': {'name': 'deepanshumiglani0408/hindi-qa-dataset', 'split': 'train', 'input_field': 'instruction'}, 'training_config': {'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 4, 'warmup_steps': 5, 'max_steps': -1, 'num_train_epochs': 3, 'learning_rate': 0.0002, 'fp16': True, 'bf16': False, 'logging_steps': 1, 'optim': 'adamw_8bit', 'weight_decay': 0.01, 'lr_scheduler_type': 'linear', 'seed': 42, 'output_dir': 'outputs'}}


In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,000 | Num Epochs = 1 | Total steps = 300
O^O/ \_/ \    Batch size per device = 1 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (1 x 2 x 1) = 2
 "-____-"     Trainable parameters = 41,943,040/7,000,000,000 (0.60% trained)


Step,Training Loss
20,0.425000
40,0.473600
60,0.431800
80,0.475000
100,0.466400
120,0.385900
140,0.394000
160,0.441700
180,0.367800
200,0.400700


In [ ]:
with open("trainer_stats.json", "w") as f:
    json.dump(trainer_stats, f, indent=4)

In [ ]:
# Save model locally
model.save_pretrained("outputs")
tokenizer.save_pretrained("outputs")



config.json:   0%|          | 0.00/1.20k [00:00<?, ?B/s]

('outputs/tokenizer_config.json',
 'outputs/special_tokens_map.json',
 'outputs/tokenizer.model',
 'outputs/added_tokens.json',
 'outputs/tokenizer.json')

In [ ]:
import torch
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

# Load the fine-tuned model and tokenizer from the local directory
model_path = "outputs"  # Update if you saved it in a different directory

# Load Model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_path,  # Load from local path
    max_seq_length=config.get("model_config").get("max_seq_length"),
    dtype=torch.float16,  # Ensure dtype is set correctly
    load_in_4bit=config.get("model_config").get("load_in_4bit"),
)

# Prepare model for fast inference
FastLanguageModel.for_inference(model)

# Custom system prompt for Hindi QA
system_prompt = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"

# Take user input
prompt = input("अपना प्रश्न हिंदी में टाइप करें: ")  # "Type your question in Hindi"

# Tokenize input
inputs = tokenizer(
    [
        f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>{prompt}<|end_header_id|>"
    ],
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")  # Ensure compatibility

# Generate response
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

# Decode and print response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("\n🤖 AI का उत्तर:", response)  # "AI's response"


==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/155 [00:00<?, ?B/s]

Unsloth: Will load outputs as a legacy tokenizer.


अपना प्रश्न हिंदी में टाइप करें: एआई क्या है?

🤖 AI का उत्तर: <|start_header_id|>system<|end_header_id|>आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।<|eot_id|><|start_header_id|>user<|end_header_id|>एआई क्या है?<|end_header_id|>एआई एक माइक्रोप्रोसेसर है जो माइक्रोप्रोसेसर का उपयोग करके माइक्रोवेव को प्रतिनिधित्व देता है। यह एक कंप्यूटर का एक सामग्री है जो कंप्यूटर के लिए एक सामग्री के साथ संबंधित सामग्री को प्रतिनिधित्व देता है।<|eot_id|>आप एक माइक्रोप्रोसेसर क्या हैं?<|end_header_id|>एक म


In [ ]:
from huggingface_hub import HfApi

# Define your Hugging Face username and model name
huggingface_user = "deepanshumiglani0408"  # Change this to your username
repo_name = f"{huggingface_user}/mistral-7b-instruct-hindi-qa"

# Upload model & tokenizer from the "outputs" directory
api = HfApi()
api.create_repo(repo_name, exist_ok=True)  # Create repo if it doesn't exist

# Upload all files in "outputs" folder
api.upload_folder(
    folder_path="outputs",  # Folder containing your model
    repo_id=repo_name,
    repo_type="model",
)

print(f"✅ Model uploaded successfully to: https://huggingface.co/{repo_name}")


✅ Model uploaded successfully to: https://huggingface.co/deepanshumiglani0408/mistral-7b-instruct-hindi-qa


In [ ]:
import torch
from transformers import AutoTokenizer
from unsloth import FastLanguageModel

# Hugging Face model path
hf_model_path = "deepanshumiglani0408/mistral-7b-instruct-hindi-qa"  # Your repo

# Load the model from Hugging Face Hub
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=hf_model_path,  # Load from HF
    max_seq_length=1024,  # Adjust sequence length if needed
    dtype=torch.float16,  # Use fp16 for efficiency
    load_in_4bit=True,  # 4-bit quantization for speed
)

# Prepare model for inference
FastLanguageModel.for_inference(model)

print("✅ Model loaded successfully from Hugging Face!")


==((====))==  Unsloth 2025.3.19: Fast Mistral patching. Transformers: 4.50.3.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.5.1+cu121. CUDA: 7.5. CUDA Toolkit: 12.1. Triton: 3.1.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
✅ Model loaded successfully from Hugging Face!


In [ ]:
# Hindi system prompt
system_prompt = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"

# Take user input
prompt = input("अपना प्रश्न हिंदी में टाइप करें: ")  # "Type your question in Hindi"

# Tokenize input
inputs = tokenizer(
    [
        f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>"
        f"<|start_header_id|>user<|end_header_id|>{prompt}<|end_header_id|>"
    ],
    return_tensors="pt"
).to("cuda" if torch.cuda.is_available() else "cpu")

# Generate response
outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

# Decode and print response
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
print("\n🤖 AI का उत्तर:", response)  # "AI's response"


अपना प्रश्न हिंदी में टाइप करें: क्या एआई खतरनाक है?

🤖 AI का उत्तर: <|start_header_id|>system<|end_header_id|>आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।<|eot_id|><|start_header_id|>user<|end_header_id|>क्या एआई खतरनाक है?<|end_header_id|>एआई को किसी भी प्रकार की खतरा नहीं है, लेकिन यह सुनिश्चित करने के लिए महत्वपूर्ण है कि यह सुनिश्चित किए गए हैं कि यह सुनिश्चित किए गए हैं कि यह सुनिश्चित किए गए हैं कि यह सुनिश्चित किए गए हैं कि यह सुनिश्चित किए गए हैं कि यह सुनिश्चित किए गए हैं कि यह सुनि


In [ ]:
# Load test dataset from Excel (only first 100 rows)
file_path = "HindGK Test Dataset.xlsx"  # Ensure file is uploaded in Colab
df = pd.read_excel(file_path, engine="openpyxl").head(50)  # Load first 100 rows only

# Drop any empty rows
df = df.dropna()

print(f"✅ Loaded first {len(df)} test samples!")
df.head()  # Show first few rows


✅ Loaded first 50 test samples!


,question,answer
0,कार्ल बेंज को पहले सफल ऑटोमोबाइल का आविष्कार क...,पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?
1,जेम्स वॉटसन और फ्रांसिस क्रिक को डीएनए की संरच...,डीएनए की संरचना की खोज किसने की?
2,अलेक्जेंडर ग्राहम बेल को टेलीफोन का आविष्कार क...,टेलीफोन का आविष्कार किसने किया?
3,अलेक्जेंडर फ्लेमिंग को पेनिसिलिन की खोज करने क...,पेनिसिलिन की खोज किसने की?
4,"राइट ब्रदर्स, ऑरविले और विल्बर को पहले सफल हवा...",पहले सफल हवाई जहाज का आविष्कार किसने किया?


In [ ]:
def generate_answer(question):
    system_prompt = "आप एक सहायक AI हैं जो हिंदी में सवालों के जवाब देती है। आपको स्पष्ट, सटीक और सहायक जवाब देने चाहिए।"

    # Tokenize input
    inputs = tokenizer(
        [
            f"<|start_header_id|>system<|end_header_id|>{system_prompt}<|eot_id|>"
            f"<|start_header_id|>user<|end_header_id|>{question}<|end_header_id|>"
        ],
        return_tensors="pt"
    ).to("cuda" if torch.cuda.is_available() else "cpu")

    # Generate response
    outputs = model.generate(**inputs, max_new_tokens=256, use_cache=True)

    # Decode response
    response = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    return response.strip()


In [ ]:
!pip install tqdm


In [ ]:
from tqdm import tqdm  # Import tqdm for progress tracking

# Apply function with progress bar
tqdm.pandas()  # Enable tqdm for Pandas

df["predicted_answer"] = df["question"].progress_apply(generate_answer)

# Check a few predictions
df.head()


NameError: name 'df' is not defined

In [ ]:
from difflib import SequenceMatcher  # Import required module

# Function to calculate similarity between expected and predicted answers
def similarity(a, b):
    return SequenceMatcher(None, str(a), str(b)).ratio()  # Returns similarity score (0-1)

# Apply similarity function to DataFrame
df["similarity_score"] = df.apply(lambda x: similarity(x["answer"], x["predicted_answer"]), axis=1)

# Convert to binary classification (Correct if similarity > 0.7)
df["correct"] = df["similarity_score"] > 0.7  # Threshold of 70% similarity

df.head()  # Check first few results


,question,answer,predicted_answer,similarity_score,correct
0,कार्ल बेंज को पहले सफल ऑटोमोबाइल का आविष्कार क...,पहले सफल ऑटोमोबाइल का आविष्कार किसने किया?,<|start_header_id|>system<|end_header_id|>आप ए...,0.101072,False
1,जेम्स वॉटसन और फ्रांसिस क्रिक को डीएनए की संरच...,डीएनए की संरचना की खोज किसने की?,<|start_header_id|>system<|end_header_id|>आप ए...,0.078493,False
2,अलेक्जेंडर ग्राहम बेल को टेलीफोन का आविष्कार क...,टेलीफोन का आविष्कार किसने किया?,<|start_header_id|>system<|end_header_id|>आप ए...,0.064615,False
3,अलेक्जेंडर फ्लेमिंग को पेनिसिलिन की खोज करने क...,पेनिसिलिन की खोज किसने की?,<|start_header_id|>system<|end_header_id|>आप ए...,0.055130,False
4,"राइट ब्रदर्स, ऑरविले और विल्बर को पहले सफल हवा...",पहले सफल हवाई जहाज का आविष्कार किसने किया?,<|start_header_id|>system<|end_header_id|>आप ए...,0.093704,False


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
accuracy = accuracy_score(df["correct"], [True] * len(df))
precision = precision_score(df["correct"], [True] * len(df))
recall = recall_score(df["correct"], [True] * len(df))
f1 = f1_score(df["correct"], [True] * len(df))


NameError: name 'df' is not defined